# Root-cause check: entity-organisation range mismatches\n\n**Author**: Sian Teesdale  \n**Date created**: 2nd July 2026  \n**Dataset Scope**: datasets present in `data/flagged_entities.csv`  \n**Purpose**: For each entity flagged in `1_find_flagged_entities.ipynb` (quality=some, provided by an active local-authority/national-park-authority/development-corporation), check whether its entity ID actually falls within an `entity-organisation.csv` range assigned to a **different** organisation than the one currently attributed on the entity table. This is the same class of bug reported in [digital-land/config#2651](https://github.com/digital-land/config/issues/2651) (entity ranges pointing at the wrong org). Entities whose ID matches their own org's range, or no range at all, are likely `quality=some` for a different reason and are lower priority.\n\nOutcome categories:\n- **different_org_range** — entity ID falls in a range assigned to another organisation → likely range/lookup misconfiguration, highest priority to report back.\n- **own_range** — entity ID falls within a range assigned to its own attributed organisation → not a range bug.\n- **no_range** — entity ID isn't covered by any range in `entity-organisation.csv` for this dataset → check `lookup.csv` separately.

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

from helpers import fetch_csv

DATA_DIR = os.path.join("..", "..", "data")
GITHUB_BASE = "https://raw.githubusercontent.com/digital-land/config/main/pipeline"

## 1. Load flagged entities from notebook 1

In [3]:
flagged_df = pd.read_csv(os.path.join(DATA_DIR, "flagged_entities.csv"))
flagged_datasets = sorted(flagged_df["dataset"].unique())
print(f"{len(flagged_df)} flagged entities across {len(flagged_datasets)} datasets")

7492 flagged entities across 13 datasets


## 2. Fetch `entity-organisation.csv` for each flagged dataset

In [4]:
print("Fetching entity-organisation.csv for flagged datasets (parallel)...")
with ThreadPoolExecutor(max_workers=10) as pool:
    futures = {
        pool.submit(fetch_csv, f"{GITHUB_BASE}/{ds}/entity-organisation.csv"): ds
        for ds in flagged_datasets
    }
    eo_parts = []
    for f in as_completed(futures):
        ds = futures[f]
        result = f.result()
        if result is None:
            print(f"  {ds}: no entity-organisation.csv")
            continue
        eo_parts.append(result)

all_eo = pd.concat(eo_parts, ignore_index=True)
all_eo.columns = [c.strip() for c in all_eo.columns]
all_eo["entity-minimum"] = pd.to_numeric(all_eo["entity-minimum"], errors="coerce")
all_eo["entity-maximum"] = pd.to_numeric(all_eo["entity-maximum"], errors="coerce")
all_eo = all_eo.dropna(subset=["entity-minimum", "entity-maximum"])
print(f"{len(all_eo)} ranges loaded across {all_eo['dataset'].nunique()} datasets")

Fetching entity-organisation.csv for flagged datasets (parallel)...
  developer-agreement-contribution: no entity-organisation.csv
  developer-agreement-transaction: no entity-organisation.csv
  developer-agreement: no entity-organisation.csv
  minerals-plan: no entity-organisation.csv
  plan-timetable: no entity-organisation.csv
  waste-plan: no entity-organisation.csv
9202 ranges loaded across 24 datasets


## 3. Match each flagged entity against ranges for its dataset

Cross-joins flagged entities with that dataset's ranges and keeps rows where the entity ID falls inside the range. Restricted to flagged datasets/entities only, so stays small enough to do per-dataset.

In [5]:
match_parts = []
for dataset, group in flagged_df.groupby("dataset"):
    ranges = all_eo[all_eo["dataset"] == dataset][["entity-minimum", "entity-maximum", "organisation"]]
    if ranges.empty:
        continue
    candidates = group[["entity", "organisation"]].rename(columns={"organisation": "attributed_organisation"})
    merged = candidates.merge(ranges, how="cross")
    matches = merged[
        (merged["entity"] >= merged["entity-minimum"]) & (merged["entity"] <= merged["entity-maximum"])
    ].copy()
    matches["dataset"] = dataset
    match_parts.append(matches)

all_matches = (
    pd.concat(match_parts, ignore_index=True)
    if match_parts
    else pd.DataFrame(columns=["entity", "attributed_organisation", "entity-minimum", "entity-maximum", "organisation", "dataset"])
)
print(f"{len(all_matches)} (entity, range) matches found")

7390 (entity, range) matches found


## 4. Classify each flagged entity

- `different_org_range` — matched a range, but for a different organisation than currently attributed
- `own_range` — matched a range for its own attributed organisation
- `no_range` — no matching range at all (check `lookup.csv` separately)

In [6]:
def _classify(group):
    range_orgs = sorted(set(group["organisation"]))
    attributed = group["attributed_organisation"].iloc[0]
    check = "own_range" if attributed in range_orgs else "different_org_range"
    return pd.Series({"range_check": check, "range_organisations": ", ".join(range_orgs)})


if not all_matches.empty:
    classified = (
        all_matches.groupby(["dataset", "entity"])
        .apply(_classify, include_groups=False)
        .reset_index()
    )
else:
    classified = pd.DataFrame(columns=["dataset", "entity", "range_check", "range_organisations"])

flagged_with_check = flagged_df.merge(classified, on=["dataset", "entity"], how="left")
flagged_with_check["range_check"] = flagged_with_check["range_check"].fillna("no_range")
flagged_with_check["range_organisations"] = flagged_with_check["range_organisations"].fillna("")

flagged_with_check["range_check"].value_counts()

range_check
own_range    7336
no_range      156
Name: count, dtype: int64

## 5. Export prioritized list

`different_org_range` rows are the most likely range/lookup misconfigurations, sorted to the top — these are the actionable ones to report back on the ticket.

In [7]:
priority_order = {"different_org_range": 0, "no_range": 1, "own_range": 2}
flagged_with_check["_priority"] = flagged_with_check["range_check"].map(priority_order)
flagged_with_check = flagged_with_check.sort_values(
    ["_priority", "dataset", "organisation", "entity"]
).drop(columns="_priority")

out_path = os.path.join(DATA_DIR, "flagged_entities_with_range_check.csv")
flagged_with_check.to_csv(out_path, index=False)
print(f"Saved {len(flagged_with_check)} rows to {out_path}")

flagged_with_check[flagged_with_check["range_check"] == "different_org_range"].head(20)

Saved 7492 rows to ../../data/flagged_entities_with_range_check.csv


,dataset,entity,name,reference,organisation,organisation_name,quality,entity_url,range_check,range_organisations
